# Step 3: expanding the rest of the abbreviations

After step 1 (rule-based) and step 2 (multiple choice from the glossary candidates), the remaining abbreviations are the ones **without a glossary entry** — mostly truncated words like `iubil.`, `procur.` or diocese adjectives that slipped through. There is no curated candidate list for them, so two other safeguards replace the candidate-membership check (see `expand_rest.py`):

1. **Corpus-mined suggestions**: words that appear *unabbreviated* somewhere in the RG and share the abbreviation's prefix are offered to the model, ranked by frequency. They are real surface forms from the same corpus, so the model can usually pick an already correctly inflected word instead of hallucinating one.
2. **Truncation constraint**: these abbreviations are cut-off words, so any expansion (suggested or free) must begin with the letters of the abbreviation itself. A doubled final consonant (`eccll.`, `diocc.`) marks a plural and is allowed to collapse (`eccll.` → `ecclesiarum`). Expansions that violate the constraint are rejected and the abbreviation is kept.

The model may also answer `"SKIP"` for tokens that are not really abbreviations (e.g. a word before a sentence-final period) or that it cannot expand confidently — keeping an abbreviation is always better than a wrong expansion. As in step 2, the occurrences are marked as `[[id|abbreviation]]`, the model returns only JSON, and the substitution happens programmatically, so the surrounding text cannot be corrupted. Every response is recorded with an acceptance tier (`suggestion` / `free` / `skip` / `rejected` / `missing`) for auditing.

In [ ]:
from helper_functions import vita_df_to_text, text_to_vita_df, call_chat_ai
from expand_rest import Vocabulary, find_remaining_occurrences, mine_candidates, build_user_prompt, parse_expansions
from multiple_choice import apply_choices
import polars as pl
from openai import OpenAI
import json

# API configuration (same setup as in expanding_candidates.ipynb)
BASE_URL = "https://chat-ai.academiccloud.de/v1"
MODEL = "gemma-4-31b-it"
api_key = ""
MAX_ROW_ATTEMPTS = 5

CLIENT = OpenAI(api_key=api_key, base_url=BASE_URL)

SYSTEM_PROMPT = """**Role:** You are a historian specializing in medieval church history with expert knowledge of the Latin abbreviations used in the papal registers.

**Task:** You will receive a Latin text in which some abbreviations are marked as `[[id|abbreviation]]`. These abbreviations are simply cut-off words (they have no entry in the abbreviation glossary). For every id, either give the full word or answer "SKIP".

**Instructions:**

1. **Truncation:** Every expansion must begin with exactly the letters of the abbreviation (the part before the period) and continue it. Example: `iubil.` may become `iubilei` but never `indulgentia`. Exception: a doubled final consonant marks a plural and collapses, e.g. `eccll.` → `ecclesiarum`, `diocc.` → `diocesium`.

2. **Suggestions:** For each id you get a list of words that occur unabbreviated in the same corpus and start with the same letters, most frequent first. Prefer one of them if it fits the grammatical and semantic context. The lists can be incomplete or misleading, so you may also answer with a fitting word that is not listed.

3. **Inflection:** Give the word in the grammatical form the context requires (case, number). If several suggested forms fit and you are unsure, take the most frequent one.

4. **SKIP:** Answer "SKIP" when the marked token is not actually an abbreviation (e.g. a complete word directly before a sentence-final period, or part of an archival reference) or when you cannot expand it with confidence. Keeping the abbreviation is better than guessing wrong.

5. **Output format:** Return only a single JSON object mapping every id to its expansion or "SKIP", e.g. `{"1": "iubilei", "2": "SKIP"}`. Do not include explanations, commentary, or any other text.
"""

The vocabulary for the suggestions is mined once from the **whole** RG (not just the Ablass subset): every word of ≥3 letters that is not itself abbreviated (i.e. not followed by a period), with its frequency.

In [3]:
# input: the output of step 2
twice_expanded = pl.read_csv("data/gemma4/twice_expanded.csv")

# corpus vocabulary from the full RG for the prefix suggestions
rg_all = pl.read_csv("data/RG_header_sublemma_all.csv").select(["header_no_tags", "regest_no_tags"])
VOCABULARY = Vocabulary.from_texts(
    rg_all.get_column("header_no_tags").drop_nulls().to_list()
    + rg_all.get_column("regest_no_tags").drop_nulls().to_list()
)
print(f"{len(VOCABULARY.words)} distinct words, {sum(VOCABULARY.counts.values())} tokens")

192229 distinct words, 1487814 tokens


In [4]:
def expand_single_rest(text: str):

    occurrences = find_remaining_occurrences(text)
    if not occurrences:
        return None

    candidates = mine_candidates(occurrences, VOCABULARY)
    user_prompt = build_user_prompt(text, occurrences, candidates)

    choices, details, errors, dump = None, [], [], None
    for attempt in range(MAX_ROW_ATTEMPTS):
        dump = call_chat_ai(CLIENT, MODEL, SYSTEM_PROMPT, user_prompt)
        content = dump["choices"][0]["message"]["content"]
        choices, details, errors = parse_expansions(content, occurrences, candidates)
        if choices is not None:
            break
    if choices is None:
        # no parseable response after all attempts -> leave everything unexpanded
        choices = {}

    return {
        "text": apply_choices(text, occurrences, choices),
        "candidates": candidates,
        "details": details,
        "errors": errors,
        "user_prompt": user_prompt,
        "response": dump,
    }

In [7]:
volume = 2
nr = 913

vita_df = twice_expanded.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr))
twice_expanded_text = vita_df_to_text(vita_df)

occurrences = find_remaining_occurrences(twice_expanded_text)
candidates = mine_candidates(occurrences, VOCABULARY)

print(twice_expanded_text)
print('-'*100 + '\n')
for occ in occurrences:
    print(f"[[{occ.id}]] {occ.matched}: {', '.join(candidates[occ.abbreviation]) or '(none)'}")

Brunswic
procons. et capitulum oppidum Halberstadensis diocesis: privilegium de non evocando 19 maior 1390 R 103.
procons. et capitulum oppidum Halberstadensis diocesis: revocatio privilegium de non evocando 27 december 1390 L 12 139.
procons. et capitulum oppidum Halberstadensis diocesis: mandatum, ut archidiaconi et officiales vicarios in dictus opido instituant 8 augustus 1391 L 10 231.
procons. et capitulum oppidum Halberstadensis diocesis: mandatum ut iam antea Rolando decanus ecclesia sancti Blasii in dictus oppidum publicus constitutio Provide attendentes " a Bonifatio VIII papa factus 7 maior 1400 L 79 224. "
mensa decanus, capitulum et singuli canonici ecclesia sancti Blasii Halberstadensis diocesis: incorporare parochialis ecclesia sancti Odalrici Brunswic. 22 aprilis 1399 L 69 185v.
parochialis ecclesia sancti Catherine Halberstadensis diocesis: indulgentia ad instar ecclesia sancti Marci de Venetiis 12 december 1399 L 82 175.
------------------------------------------------

In [8]:
result = expand_single_rest(twice_expanded_text)
if result:
    thrice_expanded_text = result["text"]
    print(thrice_expanded_text)

Brunswic
proconsul et capitulum oppidum Halberstadensis diocesis: privilegium de non evocando 19 maior 1390 R 103.
proconsul et capitulum oppidum Halberstadensis diocesis: revocatio privilegium de non evocando 27 december 1390 L 12 139.
proconsul et capitulum oppidum Halberstadensis diocesis: mandatum, ut archidiaconi et officiales vicarios in dictus opido instituant 8 augustus 1391 L 10 231.
proconsul et capitulum oppidum Halberstadensis diocesis: mandatum ut iam antea Rolando decanus ecclesia sancti Blasii in dictus oppidum publicus constitutio Provide attendentes " a Bonifatio VIII papa factus 7 maior 1400 L 79 224. "
mensa decanus, capitulum et singuli canonici ecclesia sancti Blasii Halberstadensis diocesis: incorporare parochialis ecclesia sancti Odalrici Brunswich 22 aprilis 1399 L 69 185v.
parochialis ecclesia sancti Catherine Halberstadensis diocesis: indulgentia ad instar ecclesia sancti Marci de Venetiis 12 december 1399 L 82 175.


In [9]:
for detail in result["details"]:
    print(f"[[{detail['id']}]] {detail['abbreviation']} → {detail['expansion']} ({detail['tier']})")

print()
for error in result["errors"]:
    print(error["message"])

[[1]] procons. → proconsul (suggestion)
[[2]] procons. → proconsul (suggestion)
[[3]] procons. → proconsul (suggestion)
[[4]] procons. → proconsul (suggestion)
[[5]] Brunswic. → Brunswich (suggestion)



# expanding a batch

In [16]:
results = []
thrice_expanded = []
testset_ids = twice_expanded.select("volume", "nr_RG").unique().sort(by="*")

In [17]:
i = 0

for row in testset_ids.iter_rows(named=True):

    vita_df = twice_expanded.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))
    twice_expanded_text = vita_df_to_text(vita_df)

    result = expand_single_rest(twice_expanded_text)
    if result is None:
        print(f"skipped vita #{i} (volume {row['volume']} - nr {row['nr_RG']}) with no remaining abbreviations")
        i += 1
        thrice_expanded.append(vita_df)
        continue

    thrice_expanded_text = result["text"]
    thrice_expanded.append(text_to_vita_df(thrice_expanded_text, row["volume"], row["nr_RG"]))

    results.append({
        "volume": row["volume"],
        "nr_RG": row["nr_RG"],
        "candidates": result["candidates"],
        "details": result["details"],
        "errors": result["errors"],
        "twice_expanded_text": twice_expanded_text,
        "thrice_expanded_text": thrice_expanded_text,
    })

    i += 1
    if i % 10 == 0:
        print(f"finished processing {i} vitas")

thrice_expanded = pl.concat(thrice_expanded)

skipped vita #1 (volume 2 - nr 645) with no remaining abbreviations
skipped vita #2 (volume 2 - nr 878) with no remaining abbreviations
skipped vita #4 (volume 2 - nr 916) with no remaining abbreviations
skipped vita #5 (volume 2 - nr 919) with no remaining abbreviations
skipped vita #6 (volume 2 - nr 1462) with no remaining abbreviations
skipped vita #7 (volume 2 - nr 1596) with no remaining abbreviations
skipped vita #8 (volume 2 - nr 1682) with no remaining abbreviations
skipped vita #9 (volume 2 - nr 2229) with no remaining abbreviations
skipped vita #14 (volume 2 - nr 3398) with no remaining abbreviations
skipped vita #15 (volume 2 - nr 3412) with no remaining abbreviations
skipped vita #16 (volume 2 - nr 3604) with no remaining abbreviations
skipped vita #18 (volume 2 - nr 5321) with no remaining abbreviations
skipped vita #19 (volume 2 - nr 5329) with no remaining abbreviations
skipped vita #21 (volume 2 - nr 5439) with no remaining abbreviations
skipped vita #22 (volume 2 - nr 

In [18]:
with open("data/gemma4/results_rest.json", "w") as file:
    json.dump(results, file, indent=2)

thrice_expanded.write_csv("data/gemma4/thrice_expanded.csv")

In [19]:
from collections import Counter

pattern = r"[a-zA-Z]{2,200}\."

def count_abbreviations(df):
    counts = df.select(
        pl.col("header_no_tags").str.count_matches(pattern).sum(),
        pl.col("regest_no_tags").str.count_matches(pattern).sum(),
    )
    return counts.row(0)[0] + counts.row(0)[1]

print(f"abbreviations twice: {count_abbreviations(twice_expanded)}")
print(f"abbreviations thrice: {count_abbreviations(thrice_expanded)}")
print()

tiers = Counter(detail["tier"] for result in results for detail in result["details"])
for tier, count in tiers.most_common():
    print(f"{tier}: {count}")

abbreviations twice: 69
abbreviations thrice: 3

suggestion: 45
free: 21
rejected: 2
skip: 1
